# 2 · Sentiment Analysis — Model Training

1. Load preprocessed padded sequences from `artifacts/sentiment/`
2. Build Bidirectional LSTM model
3. Train with early stopping, LR scheduling, and model checkpointing
4. Save the best model → `models/sentiment/sentiment_model.keras`
5. Plot training curves

## 0 · Configuration

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pickle
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from src.config import (
    SENTIMENT_TRAIN_ARTIFACT,
    SENTIMENT_TEST_ARTIFACT,
    SENTIMENT_TOKENIZER_PATH,
    SENTIMENT_LABEL_ENC_PATH,
    SENTIMENT_MODEL_PATH,
    SENT_VOCAB_SIZE,
    SENT_EMBEDDING_DIM,
    SENT_LSTM_UNITS,
    SENT_DROPOUT_RATE,
    SENT_BATCH_SIZE,
    SENT_EPOCHS,
    SENT_MAX_LEN,
    SENTIMENT_LABELS,
)
from src.common import set_seed, configure_gpu
from src.sentiment.model import build_sentiment_model, get_callbacks
from src.sentiment.utils import plot_training_history
from src.sentiment.preprocessing import load_tokenizer, load_label_encoder

set_seed(42)
configure_gpu()
print(f'TensorFlow {tf.__version__}')

## 1 · Load Artifacts

In [ ]:
with open(SENTIMENT_TRAIN_ARTIFACT, 'rb') as fh:
    train_data = pickle.load(fh)
with open(SENTIMENT_TEST_ARTIFACT, 'rb') as fh:
    test_data  = pickle.load(fh)

X_train = train_data['X']
y_train = train_data['y']
X_test  = test_data['X']
y_test  = test_data['y']

tokenizer     = load_tokenizer(SENTIMENT_TOKENIZER_PATH)
label_encoder = load_label_encoder(SENTIMENT_LABEL_ENC_PATH)

NUM_CLASSES  = len(label_encoder.classes_)
ACTUAL_VOCAB = min(SENT_VOCAB_SIZE, len(tokenizer.word_index) + 1)

print(f'Train  : {X_train.shape}  |  Test: {X_test.shape}')
print(f'Classes: {list(label_encoder.classes_)}  (n={NUM_CLASSES})')
print(f'Vocab  : {ACTUAL_VOCAB:,}')

## 2 · Build Model

In [ ]:
model = build_sentiment_model(
    vocab_size    = ACTUAL_VOCAB,
    embedding_dim = SENT_EMBEDDING_DIM,
    lstm_units    = SENT_LSTM_UNITS,
    dropout_rate  = SENT_DROPOUT_RATE,
    num_classes   = NUM_CLASSES,
    max_len       = SENT_MAX_LEN,
)

model.summary()

## 3 · Train

In [ ]:
callbacks = get_callbacks(SENTIMENT_MODEL_PATH)

history = model.fit(
    X_train, y_train,
    validation_data = (X_test, y_test),
    epochs          = SENT_EPOCHS,
    batch_size      = SENT_BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

print(f'\nBest val accuracy: {max(history.history["val_accuracy"]):.4f}')

## 4 · Training Curves

In [ ]:
fig = plot_training_history(history)
fig.savefig('sentiment_training_history.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved to sentiment_training_history.png')

## 5 · Quick Test on Sample Texts

In [ ]:
from src.sentiment.predict import full_analysis

samples = [
    'I absolutely love this product! It is amazing and works perfectly.',
    'This is the worst experience I have ever had. Totally disappointed.',
    'The product arrived on time. It is okay, nothing special.',
    'Feeling very anxious and worried about the upcoming exams.',
    'I am so excited and thrilled about the new opportunity!',
]

print('Quick Inference Check')
print('=' * 60)
for text in samples:
    result = full_analysis(text)
    print(f'Text     : {text[:60]}...' if len(text) > 60 else f'Text     : {text}')
    print(f'Sentiment: {result["label"]}  (conf: {result["confidence"]*100:.1f}%)')
    print(f'Emotion  : {result["description"]}')
    print()

## Training Complete

Model saved to:
- `models/sentiment/sentiment_model.keras`
